# English → Russian: train the scratch Transformer

Stage 1 training notebook, not the future combined notebook. Choose **Runtime → Change runtime type → T4 GPU**, then run top to bottom. The raw dataset is already in Git; no Kaggle download or API is used. GPU training has not been run by the implementation author. Budget multiple Colab sessions; checkpoint after every epoch. A disconnect during an epoch replays that epoch.

## 1. Clone the repository
Use the implementation branch before the PR is merged; change `REPO_REF` to `main` after merge. A fresh Colab runtime has no checkout; an existing checkout is reused without reset.

In [ ]:
from pathlib import Path
import subprocess
REPO_URL = "https://github.com/vcdx1111/Machine-translation-English-to-Russian-.git"
REPO_REF = "feat/scratch-transformer"
REPO = Path("/content/Machine-translation-English-to-Russian-")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
%cd /content/Machine-translation-English-to-Russian-
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 2. Install pinned packages
Install into the notebook kernel environment. If Colab asks for a runtime restart, restart and rerun from the top. The command fails immediately if installation fails.

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime before full training"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount Drive and choose a run
Checkpoints and results persist on Google Drive. Keep the same run name to resume; use a new name for a fresh experiment. Set `RESUME=True` after a disconnect. Only resume checkpoints you created and trust. Training restores model, optimizer, scaler, Noam step, early-stopping and RNG state at the last completed epoch. Do not run two sessions writing the same run directory.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_NAME = "scratch_transformer_seed42"
RESUME = False
RUN = Path("/content/drive/MyDrive/en_ru_translation") / RUN_NAME
CHECKPOINTS = RUN / "checkpoints"
RESULTS = RUN / "results"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
if RESUME:
    assert (CHECKPOINTS / "last.pt").exists(), "No last.pt for this run"

## 4. Load and audit the checked-in raw TSV
Read all three columns and create source-grouped train/validation/test splits. The same normalization and split seed are used during training. SentencePiece sees train pairs only. Inspect the measured report before training.

In [ ]:
from src.data import prepare, read_raw
RAW = REPO / "data/raw/rus.txt"
assert RAW.is_file(), "Expected the tracked data/raw/rus.txt"
rows, widths, malformed = read_raw(RAW)
print("Rows:", len(rows), "Columns:", widths, "Malformed:", malformed)
print(rows[:2])
prepare(RAW, "data/processed/full", seed=42)
import json
print(json.dumps(json.loads(Path("data/processed/full/audit.json").read_text()), ensure_ascii=False, indent=2))
del rows

## 5. Train (or resume)
Use the six-layer, 512-dimensional base configuration. AMP is enabled on CUDA; validation loss selects the best epoch. Training automatically evaluates the best weights on test once the run finishes. Do not select hyperparameters by test BLEU. If T4 runs out of memory, use a **new run name** with a copied YAML and smaller batch size; document this change. Checkpoints are saved each epoch, not each batch.

Planning estimate, not measured: roughly 12–48 GPU-hours for 30 epochs plus 0.5–3 hours for serial beam evaluation on T4; hardware and sentence lengths can change this substantially. Estimate your actual total from the first epoch log, allow early stopping, and resume across sessions.

In [ ]:
CONFIG = "configs/transformer_base.yaml"
command = [sys.executable, "-m", "src.train", "--config", CONFIG,
           "--checkpoint-dir", str(CHECKPOINTS), "--results-dir", str(RESULTS)]
if RESUME:
    command += ["--resume", str(CHECKPOINTS / "last.pt")]
subprocess.run(command, check=True)

## 6. Standalone evaluation (optional rerun)
Training already writes test metrics and translations. Run this only if evaluation was interrupted or to reproduce it. The checkpoint bundles its own tokenizer and best weights; the checked-in raw dataset must match its hash. Use a separate output directory when comparing greedy with beam search.

In [ ]:
RERUN_EVALUATION = False
if RERUN_EVALUATION:
    subprocess.run([sys.executable, "-m", "src.evaluate", "--checkpoint", str(CHECKPOINTS / "last.pt"),
                    "--results-dir", str(RESULTS)], check=True)

## 7. Inspect and share results
Send `metrics.json`, `train_log.csv`, `samples.txt`, the exact config and Git commit, GPU model, elapsed time, and any errors. Keep checkpoints/tokenizer on Drive; do not commit model weights. The next decision (improve scratch model or fine-tune Helsinki) follows the README criteria. No Helsinki fine-tuning or combined notebook is included yet.

In [ ]:
import csv
metrics = json.loads((RESULTS / "metrics.json").read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print((RESULTS / "train_log.csv").read_text())
print((RESULTS / "samples.txt").read_text())
assert len(list(csv.DictReader((RESULTS / "train_log.csv").open()))) > 0
assert (RESULTS / "samples.txt").read_text().count("EN: ") == 30
print("Results saved in:", RESULTS)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)